<a href="https://colab.research.google.com/github/Ma5nu/AV2/blob/main/Atividade_Pr%C3%A1tica_Explorando_Dados_na_Web_com_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Nível 1: Básico (URLs, Parâmetros e Cabeçalhos)

### Exercício 1.1, 1.2, 1.3 e 1.4

Este exercício demonstra como fazer uma requisição GET para uma API pública (`JSONPlaceholder`), utilizando parâmetros para filtrar os resultados, incluindo cabeçalhos personalizados e verificando o status da resposta e a URL final.

In [ ]:
import requests

# URL da API do JSONPlaceholder para posts
url = "https://jsonplaceholder.typicode.com/posts"

# Exercício 1.2: Utilizar o argumento params para filtrar os resultados
# Buscando apenas posts do userId igual a 2
params = {
    'userId': 2
}

# Exercício 1.3: Incluir um dicionário de cabeçalhos (headers)
# Passando um User-Agent personalizado
headers = {
    'User-Agent': 'MeuProjetoDeExploracaoDeDados/1.0'
}

# Exercício 1.1 e 1.4: Faça uma requisição GET e imprima o status_code e a URL final
# Lembre-se de sempre utilizar o parâmetro timeout
try:
    response = requests.get(url, params=params, headers=headers, timeout=10)

    # Exercício 1.4: Imprimir o código de status HTTP e a URL final
    print(f"Código de Status HTTP: {response.status_code}")
    print(f"URL Final da Requisição: {response.url}")

    # Verificar se a requisição foi bem-sucedida (status 200)
    if response.status_code == 200:
        print("Requisição realizada com sucesso!")
        # Opcional: imprimir parte do conteúdo para verificar
        # print("Conteúdo (primeiros 500 caracteres):")
        # print(response.text[:500])
    else:
        print(f"Erro na requisição. Status: {response.status_code}")

except requests.exceptions.Timeout:
    print("A requisição excedeu o tempo limite.")
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro na requisição: {e}")

Código de Status HTTP: 200
URL Final da Requisição: https://jsonplaceholder.typicode.com/posts?userId=2
Requisição realizada com sucesso!


## Nível 2: Intermediário (JSON, Erros e Arquivos Binários)

### Exercício 2.1

Este exercício demonstra como criar uma lista de CEPs, fazer um loop para consultá-los na API do ViaCEP, converter as respostas para JSON e armazenar os resultados em um DataFrame do Pandas.

### Exercício 3.3

Este exercício salva os dados extraídos dos livros (título e preço) em um arquivo CSV utilizando a biblioteca Pandas.

In [ ]:
import pandas as pd

# Verifique se a variável `extracted_books` da célula anterior existe
# Caso contrário, você pode precisar executar a célula do Exercício 3.2 novamente.
if 'extracted_books' in globals() and extracted_books:
    df_books = pd.DataFrame(extracted_books)
    csv_file_name = "livros_raspagens.csv"
    df_books.to_csv(csv_file_name, index=False, encoding='utf-8')
    print(f"Dados dos livros salvos com sucesso em '{csv_file_name}'")
    print("\nConteúdo do DataFrame (primeiras 5 linhas):")
    print(df_books.head())
else:
    print("A lista 'extracted_books' não foi encontrada ou está vazia. Por favor, execute o Exercício 3.2 novamente para extrair os dados.")

Dados dos livros salvos com sucesso em 'livros_raspagens.csv'

Conteúdo do DataFrame (primeiras 5 linhas):
                                   Title    Price
0                   A Light in the Attic  Â£51.77
1                     Tipping the Velvet  Â£53.74
2                             Soumission  Â£50.10
3                          Sharp Objects  Â£47.82
4  Sapiens: A Brief History of Humankind  Â£54.23


### Exercício 3.4

Este exercício demonstra como usar `pandas.read_html()` para capturar diretamente uma tabela de uma página da Wikipedia e transformá-la em um DataFrame, sem a necessidade de BeautifulSoup. A função `io.StringIO()` é usada para simular um arquivo em memória com o conteúdo HTML, permitindo que `read_html` o processe.

In [ ]:
import requests
from bs4 import BeautifulSoup

# URL do site para webscraping
url_scraping = "https://books.toscrape.com/"

print(f"Acessando o site para raspagem: {url_scraping}")

try:
    # Use a função download_seguro que criamos anteriormente para obter a resposta
    # Necessário definir download_seguro se ainda não foi executado
    try:
        response = download_seguro(url_scraping) # Reutilizando a função do Exercício 2.2
    except NameError:
        print("Função 'download_seguro' não definida. Definindo agora...")
        # Se download_seguro não está no escopo, defina-a novamente para esta execução
        def download_seguro(url, timeout=10):
            try:
                print(f"Tentando baixar dados de: {url}")
                response = requests.get(url, timeout=timeout)
                response.raise_for_status()
                print(f"Download bem-sucedido de {url}. Status: {response.status_code}")
                return response
            except requests.exceptions.HTTPError as http_err:
                print(f"Erro HTTP ao baixar {url}: {http_err} (Status: {response.status_code})")
                return None
            except requests.exceptions.ConnectionError as conn_err:
                print(f"Erro de Conexão ao baixar {url}: {conn_err}")
                return None
            except requests.exceptions.Timeout:
                print(f"Tempo limite excedido ao baixar {url}. Tentou dentro de {timeout} segundos.")
                return None
            except requests.exceptions.RequestException as req_err:
                print(f"Ocorreu um erro inesperado na requisição ao baixar {url}: {req_err}")
                return None
        response = download_seguro(url_scraping)


    if response:
        # Analise o conteúdo HTML com BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        # Localize os elementos da página para os livros
        # Livros estão dentro de artigos com a classe 'product_pod'
        books = soup.find_all('article', class_='product_pod')

        print("\nExtraindo dados dos 5 primeiros livros:")
        global extracted_books # Declare como global para ser acessível por outras células
        extracted_books = []

        # Extraia o título e o preço dos 5 primeiros livros
        for i, book in enumerate(books[:5]): # Limitar aos 5 primeiros
            title = book.h3.a['title']
            price = book.find('p', class_='price_color').text.strip()

            extracted_books.append({'Title': title, 'Price': price})
            print(f"Livro {i+1}: Título = '{title}', Preço = '{price}'")


    else:
        print("Não foi possível obter a página para raspagem.")

except Exception as e:
    print(f"Ocorreu um erro durante o web scraping: {e}")

Acessando o site para raspagem: https://books.toscrape.com/
Tentando baixar dados de: https://books.toscrape.com/
Download bem-sucedido de https://books.toscrape.com/. Status: 200

Extraindo dados dos 5 primeiros livros:
Livro 1: Título = 'A Light in the Attic', Preço = 'Â£51.77'
Livro 2: Título = 'Tipping the Velvet', Preço = 'Â£53.74'
Livro 3: Título = 'Soumission', Preço = 'Â£50.10'
Livro 4: Título = 'Sharp Objects', Preço = 'Â£47.82'
Livro 5: Título = 'Sapiens: A Brief History of Humankind', Preço = 'Â£54.23'


In [ ]:
import pandas as pd
import requests
import io

# URL de uma página da Wikipedia com uma tabela de dados (exemplo: Lista de países por área)
# Você pode escolher outra página com uma tabela se preferir.
wikipedia_url = "https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_%C3%A1rea"

print(f"Acessando a página da Wikipedia: {wikipedia_url}")

try:
    # Adicionar um User-Agent para simular um navegador e evitar o erro 403
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    # Faça a requisição GET para obter o conteúdo HTML da página com os headers
    response = requests.get(wikipedia_url, headers=headers, timeout=15)
    response.raise_for_status() # Verifica se a requisição foi bem-sucedida

    # Use io.StringIO para que pandas.read_html() possa ler o conteúdo como um arquivo
    html_content = io.StringIO(response.text)

    # Use pandas.read_html() para extrair as tabelas da página
    # Isso retorna uma lista de DataFrames, pois pode haver múltiplas tabelas
    tables = pd.read_html(html_content)

    print(f"Foram encontradas {len(tables)} tabelas na página.\n")

    # Vamos exibir as primeiras linhas da primeira tabela encontrada como exemplo
    if tables:
        df_wikipedia = tables[0] # Pega a primeira tabela
        print("DataFrame da primeira tabela da Wikipedia (primeiras 5 linhas):")
        print(df_wikipedia.head())

        # Opcional: Para verificar todas as tabelas, você pode iterar sobre 'tables'
        # for i, df in enumerate(tables):
        #     print(f"\n--- Tabela {i+1} ---")
        #     print(df.head())
    else:
        print("Nenhuma tabela encontrada na página.")

except requests.exceptions.Timeout:
    print(f"Erro: A requisição para {wikipedia_url} excedeu o tempo limite.")
except requests.exceptions.RequestException as e:
    print(f"Erro ao acessar a página da Wikipedia: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado ao processar a página: {e}")

Acessando a página da Wikipedia: https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_%C3%A1rea
Foram encontradas 8 tabelas na página.

DataFrame da primeira tabela da Wikipedia (primeiras 5 linhas):
  Ordem            País                           Área (km²)[nota 1]  \
0     1          Rússia                                   17 098 246   
1     2          Canadá                                    9 984 670   
2     3           China                                    9 596 961   
3     4  Estados Unidos  7 824 535, com Alasca e Havaí, 9 371 174[2]   
4     5          Brasil                                    8 510 417   

                                         Observações  
0  Não inclui a península da Crimeia. Antes da su...  
1  É o maior país em extensão territorial do cont...  
2  Sem as regiões administrativas especiais de Ho...  
3  Inclui apenas os 50 estados, entre os quais, A...  
4  Mais especificamente, segundo o IBGE (2021), a...  


In [ ]:
import requests
import pandas as pd

# Exercício 2.1: Criar uma lista com três CEPs diferentes
ceps = ["01001000", "20040030", "70070100"]

# Lista para armazenar os dados de cada CEP
dados_ceps = []

print("Consultando a API do ViaCEP para cada CEP...")

for cep in ceps:
    url_viacep = f"https://viacep.com.br/ws/{cep}/json/"

    try:
        # Faça a requisição GET com timeout
        response = requests.get(url_viacep, timeout=10)
        response.raise_for_status() # Lança exceção para códigos de status HTTP de erro

        # Converta a resposta para JSON
        data = response.json()
        dados_ceps.append(data)
        print(f"CEP {cep}: Dados coletados com sucesso.")

    except requests.exceptions.Timeout:
        print(f"CEP {cep}: A requisição excedeu o tempo limite.")
    except requests.exceptions.RequestException as e:
        print(f"CEP {cep}: Ocorreu um erro na requisição: {e}")
    except ValueError:
        print(f"CEP {cep}: Não foi possível decodificar JSON da resposta.")

# Armazene os resultados em um DataFrame do Pandas
if dados_ceps:
    df_ceps = pd.DataFrame(dados_ceps)
    print("\nDataFrame com os dados dos CEPs:")
    print(df_ceps.head())
else:
    print("Nenhum dado de CEP foi coletado para criar o DataFrame.")

Consultando a API do ViaCEP para cada CEP...
CEP 01001000: Dados coletados com sucesso.
CEP 20040030: Dados coletados com sucesso.
CEP 70070100: Dados coletados com sucesso.

DataFrame com os dados dos CEPs:
         cep      logradouro              complemento unidade  bairro  \
0  01001-000     Praça da Sé               lado ímpar              Sé   
1  20040-030  Rua do Ouvidor  de 50 ao fim - lado par          Centro   
2        NaN             NaN                      NaN     NaN     NaN   

       localidade   uf          estado   regiao     ibge   gia  ddd siafi  \
0       São Paulo   SP       São Paulo  Sudeste  3550308  1004   11  7107   
1  Rio de Janeiro   RJ  Rio de Janeiro  Sudeste  3304557         21  6001   
2             NaN  NaN             NaN      NaN      NaN   NaN  NaN   NaN   

   erro  
0   NaN  
1   NaN  
2  true  


### Exercício 2.2

Este exercício foca na criação de uma função robusta para download, que utiliza blocos `try/except` para capturar e lidar com diversos tipos de erros, incluindo erros HTTP com `response.raise_for_status()`.

In [ ]:
import requests

def download_seguro(url, timeout=10):
    """
    Faz uma requisição GET segura para uma URL e retorna o objeto response.
    Captura erros HTTP (4xx, 5xx) e outros erros de requisição.
    """
    try:
        print(f"Tentando baixar dados de: {url}")
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # Lança uma exceção para códigos de status HTTP de erro (4xx ou 5xx)
        print(f"Download bem-sucedido de {url}. Status: {response.status_code}")
        return response
    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP ao baixar {url}: {http_err} (Status: {response.status_code})")
        return None
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Erro de Conexão ao baixar {url}: {conn_err}")
        return None
    except requests.exceptions.Timeout:
        print(f"Tempo limite excedido ao baixar {url}. Tentou dentro de {timeout} segundos.")
        return None
    except requests.exceptions.RequestException as req_err:
        print(f"Ocorreu um erro inesperado na requisição ao baixar {url}: {req_err}")
        return None

# Exemplos de uso da função:
print("\n--- Teste com URL válida ---")
valid_url = "https://jsonplaceholder.typicode.com/posts/1"
resposta_valida = download_seguro(valid_url)
if resposta_valida:
    print("Conteúdo (primeiros 100 caracteres):", resposta_valida.text[:100], "...")

print("\n--- Teste com URL que causa erro 404 ---")
# Esta URL deve retornar um erro 404 (Not Found)
invalid_url = "https://jsonplaceholder.typicode.com/posts/9999999999999999"
resposta_invalida = download_seguro(invalid_url)
if resposta_invalida is None:
    print("Não foi possível baixar dados da URL inválida (como esperado).")

print("\n--- Teste com URL que não existe (provável erro de conexão/domínio) ---")
non_existent_url = "https://this-domain-does-not-exist-123456789.com"
resposta_nao_existente = download_seguro(non_existent_url)
if resposta_nao_existente is None:
    print("Não foi possível baixar dados da URL inexistente (como esperado).")


--- Teste com URL válida ---
Tentando baixar dados de: https://jsonplaceholder.typicode.com/posts/1
Download bem-sucedido de https://jsonplaceholder.typicode.com/posts/1. Status: 200
Conteúdo (primeiros 100 caracteres): {
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio ...

--- Teste com URL que causa erro 404 ---
Tentando baixar dados de: https://jsonplaceholder.typicode.com/posts/9999999999999999
Erro HTTP ao baixar https://jsonplaceholder.typicode.com/posts/9999999999999999: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/9999999999999999 (Status: 404)
Não foi possível baixar dados da URL inválida (como esperado).

--- Teste com URL que não existe (provável erro de conexão/domínio) ---
Tentando baixar dados de: https://this-domain-does-not-exist-123456789.com
Erro de Conexão ao baixar https://this-domain-does-not-exist-123456789.com: HTTPSConnectionPool(host='this-domain-does-not-exist-123456789.com'

### Exercício 2.3

Este exercício demonstra como baixar uma imagem aleatória de uma URL e salvá-la em um arquivo local em modo binário (`'wb'`).

In [ ]:
import requests
import os

# URL para baixar uma imagem aleatória
image_url = "https://picsum.photos/400/400"

# Nome do arquivo para salvar a imagem
file_name = "imagem_aleatoria.jpg"

print(f"Tentando baixar a imagem de: {image_url}")

try:
    # Faça a requisição GET para a imagem
    # stream=True permite baixar o conteúdo em pedaços, útil para arquivos grandes
    response = requests.get(image_url, stream=True, timeout=10)
    response.raise_for_status()  # Verifica se a requisição foi bem-sucedida

    # Salve o conteúdo bruto da resposta em um arquivo local no modo de escrita em bytes ('wb')
    with open(file_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"Imagem salva com sucesso como '{file_name}' no diretório atual.")
    # Verifique se o arquivo foi criado e seu tamanho (opcional)
    if os.path.exists(file_name):
        print(f"Tamanho do arquivo: {os.path.getsize(file_name)} bytes")

except requests.exceptions.Timeout:
    print(f"Erro: A requisição excedeu o tempo limite ao tentar baixar a imagem.")
except requests.exceptions.RequestException as e:
    print(f"Erro ao baixar a imagem: {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Tentando baixar a imagem de: https://picsum.photos/400/400
Imagem salva com sucesso como 'imagem_aleatoria.jpg' no diretório atual.
Tamanho do arquivo: 31823 bytes
